# آزمایش سریع کد (بدون GPU)

این دفترچه فقط بررسی می‌کند که **همه‌ی مراحل بدون خطا اجرا می‌شوند**. نتایج آن معنی ندارند.
- مدل کوچک **mT5-small**، فقط **۲۴ نمونه**، **۱ ایپاک**
- خروجی در پوشه‌ی جدا `smoke_outputs/` (به Drive و اجرای اصلی کاری ندارد)
- **بدون GPU** اجرا می‌شود: اگر پیام محدودیت GPU آمد، **Connect without GPU** را بزنید
- زمان: حدود ۱۵ تا ۲۵ دقیقه

**Runtime → Run all**

In [ ]:
import os, glob, zipfile, shutil
os.environ['SMOKE'] = '1'                 # smoke-test mode (see utils.load_config)
REPO = '/content/persian-mt5-summarization-sentiment'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/zph272/persian-mt5-summarization-sentiment.git {REPO}
%cd {REPO}
!git pull -q
os.makedirs('data/processed', exist_ok=True)
for zp in glob.glob('*.zip'):
    zipfile.ZipFile(zp).extractall('data/processed')
if not os.path.exists('data/processed/joint_clean.csv'):
    shutil.copy('data/processed/joint_train.csv', 'data/processed/joint_clean.csv')
shutil.rmtree('smoke_outputs', ignore_errors=True)
print(sorted(os.listdir('data/processed')))
!pip install -q rouge_score sentencepiece

## مرحله ۱، ۲، ۳

In [ ]:
!python 4_train_stage1_summ.py

In [ ]:
!python 5_train_stage2_sent.py

In [ ]:
!python 6_train_stage3_joint.py

## ارزیابی، ablation، نمودار

In [ ]:
!python 7_evaluate.py --baselines
!python 7_evaluate.py --model summ_final  --tasks summ,loss
!python 7_evaluate.py --model sent_final  --tasks sent
!python 7_evaluate.py --model joint_final --tasks summ,sent,loss
!python 9_ablation.py --only sent

In [ ]:
!python 8_plot_results.py

## نتیجه آزمایش

In [ ]:
expected = ['models/summ_final', 'models/sent_final', 'models/joint_final', 'models/sent_only',
            'results/summ_training_history.csv', 'results/sent_final_training_history.csv',
            'results/joint_final_training_history.csv', 'results/baselines_eval.json',
            'results/summ_final_eval.json', 'results/sent_final_eval.json', 'results/joint_final_eval.json',
            'results/sent_only_eval.json', 'results/results_table.md', 'figures/loss_curves.png']
missing = [e for e in expected if not os.path.exists('smoke_outputs/' + e)]
print('❌ MISSING (send a screenshot of the error above):', missing) if missing else print('✅ ALL STEPS RAN WITHOUT ERRORS')